# 金融數據工作坊 - 資料管理

## 工作坊系列回顧

在上一場（爬蟲）結束後，我們手上多了一堆 csv 檔（`NVDA.csv`、`AAPL.csv`⋯），裡面是每檔股票的歷史每日交易紀錄。  
這場要解決的問題是拿到這些資料後怎麼存、怎麼查、怎麼分享。

## 資料管理思維

- **儲存**：用什麼結構整理資料？(資料庫設計)
- **存取**：怎麼查、怎麼新增、修改、刪除？(資料庫語法)
- **分享**：怎麼把資料提供給對的對象？(網頁、API、圖表⋯)

> 💡 「分享」有很多種形式，全看**呈現對象是誰、他們想看到怎樣的資料**：
> - 給工程師 / 其他系統 → 結構化資料（API、JSON）
> - 給一般使用者 → 網頁 / App
> - 給決策者 / 分析師 → 圖表、儀表板
>
> 它們**共用同一個資料源**，只是呈現方式不同。
> 也就是說有好的資料管理，同一份資料源就可以提供多元的呈現方式。


# SQL

金融數據工作坊系列活動中，我們透過爬蟲把股價資料抓下來，存成 CSV 檔保存。  
但是當我們想查詢「NVIDIA 2026-04-23 之後每天的開盤價」、或「成交量大於 110,000,000 的那幾天」，  
就得每次重新寫 Python 讀檔、過濾資料，既麻煩又重複。

如果資料能存放在一個**可以直接發問**的系統裡，事情會簡單很多，這就是**資料庫 (Database)**。  
而 **SQL (Structured Query Language)**，就是我們用來跟資料庫對話的語言。

## 為什麼要用資料庫？
- **隨時隨地存取**：新增、查詢、修改、刪除，都只要一句 SQL
- **處理大量資料更有效率**：資料量變大時，資料庫會用索引(Index)等技術加速，遠快過每次都重新讀整份檔案
- **方便和各種服務系統串接**：網站、App、分析工具都可以共用同一份資料
- **資料一致性**：避免同一份資料被複製到多個檔案、每次修改都要手動同步

### 那為什麼不用 Excel / Google Sheet？
試算表對小量、需要手動操作的資料非常好用：
- 內建函式（SUM, VLOOKUP, QUERY…）、能畫圖表、也能多人共編
- 很適合臨時分析、報表呈現、財務試算

但當資料量變大、或想讓**程式 / 服務自動取用**時，就會遇到幾個瓶頸：
- **資料量一大就變慢**：開檔、捲動、公式重算都明顯卡頓，甚至直接打不開
- **多人 / 多程式同時寫入容易出事**：同一格互相覆蓋、衝突，難以追蹤誰改了什麼
- **程式存取不直接**：每次都要先匯入 / 匯出成檔案，格式還常跑掉（日期變文字、`0050` 開頭的 0 消失…）
- **資料型別沒有保障**：同一欄可以塞任何東西，一不小心就髒掉
- **跨分頁查詢不直覺**：多個分頁要用 VLOOKUP 串，條件一複雜就非常難寫

> 這些「資料量、同時寫入、程式介接、型別一致、多表串接」的問題，正是**關聯式資料庫**擅長的處理範圍。

## 資料庫種類
- **關聯式資料庫 (SQL)**：資料欄位須先定義清楚、結構穩定，適合帳務、訂單、股價這類格式固定的資料
- **非關聯式資料庫 (NoSQL)**：資料擁有彈性結構、讀寫速度較快，適合留言、貼文、感測器紀錄這類結構多變或即時性的資料

> 兩者並非二選一，實務上大型系統常常同時使用：核心交易資料放關聯式、快取或 log 放非關聯式。

## 常見的關聯式資料庫種類
- MySQL：免費、開源性質、常應用於Web，較適合中小型企業事務
- PostgreSQL：免費、開源、以商業應用導向為主
- MSSQL：Microsoft SQL Server，only on Windows，花錢就能買的完整方案。
- Oracle：公部門常用的資料庫系統，安全、效能、功能都極為齊全。
- SQLite：基於文件的資料庫，不需要架設伺服器，就能處理各種類數據。

## 常見的非關聯式資料庫種類
NoSQL 依「資料模型」可以再分成幾大類：

- **文件型 (Document)**：**MongoDB**、**Couchbase**
    - 用類似 JSON 的結構存取，欄位可彈性增減，常用於內容管理、使用者資料
- **鍵值 (Key-Value)**：**Redis**、**memcache**
    - 極快的讀寫，常用於購物車紀錄、登入狀態 (session)、加速網站讀取效率 (cache)
- **欄族型 (Column-family)**：**Cassandra**、**HBase**
    - 適合大量分散式寫入，常用於系統日誌、感測器等訊息紀錄
- **圖型 (Graph)**：**Neo4J**
    - 擅長處理節點之間的關聯，常用於社交網路、推薦系統、風控關係鏈

## 資料庫常見操作情境

要把資料放進資料庫並使用，通常會經過這幾個階段：

1. **建立資料庫 (Database)** — 建一個檔案或服務來存資料  

2. **建立資料表 (Table)** — 規劃資料的結構，資料該分別存在哪些表，有哪些欄位。  

3. **日常操作 (CRUD)** — 對資料表裡的資料做四件事
    - **C**reate（新增）：抓到一天新的股價，就寫入一筆紀錄
    - **R**ead  （讀取）：查 NVIDIA 最近 5 天的收盤價
    - **U**pdate（更新）：修正寫錯的數字
    - **D**elete（刪除）：清掉異常的資料

> 前兩項是「**一次性的設定**」（只要在一開始做一次），第三項是「**長期反覆的操作**」。  
> 大部分時間我們其實都是在做 CRUD。


## 在建表前先想：要建幾張表？

在動手建立資料表之前，先想一下我們未來會想對這份資料做什麼事：

- 「查 NVIDIA 上週的收盤價」 → 需要每日交易數字
- 「記錄股票的公司名」→ 需要股票的代號及公司名稱資訊
- 「列出所有半導體類股」 → 需要股票的產業歸屬

仔細看會發現這些需求其實混著兩種資訊：

| 層次 | 例子 | 變動頻率 |
|---|---|---|
| **股票本身的屬性** | NVDA → NVIDIA、所屬產業 | 幾乎不變 |
| **每天的交易紀錄** | 2026-04-20 收盤 202.06 | 每天新增 |

原始 csv 其實只有「每天的交易紀錄」，但我們匯入時會補上股票代號、後續也會想加更多股票本身的欄位（公司名稱、產業類別...）。  
這兩種資訊的「變動頻率」差很多，都放入同一張表在未來的變動可能會有管理不易的狀況：

- 每筆每日資料都要重複存「NVIDIA」這四個字 — 5 年 × 250 個交易日 = 1250 次重複。  
- 公司若改名要全部 UPDATE，手誤打錯一筆就資料不一致。

所以分成兩張表，用股票代號 `stock_id` 對應起來：

- 📈 `stock_date`：每天的交易紀錄
- 🏷️ `stock_list`：股票本身（代號、名稱...）

> 後面會學到 SQL 有語法提供多表串接。


### 那為什麼不是 NVDA、AAPL 各一張表？

我們手上是 `NVDA.csv`、`AAPL.csv` 兩個獨立檔案，最直覺的反應是：那資料庫裡也開 `NVDA`、`AAPL` 兩張表吧？  
但這其實是把「檔案系統的限制」誤當成「資料庫的設計」。

正確的做法是**所有股票共用一張 `stock_date`**，用 `stock_id` 欄位區分是哪檔股票。原因如下：

| 觀點 | 一檔一張表 (NVDA / AAPL / ...) | 全部共用 `stock_date` |
|---|---|---|
| **股票代號的角色** | 變成「資料表的名字」 | 是「欄位裡的內容」 ✅ |
| **新增一檔股票** | 每加一支就要先「新建一張表」 | 把資料寫進現成的表就好 ✅ |
| **跨股票比較**<br>（例：當天漲幅前 5 大） | 要一張一張查，再把結果拼起來 | 全部資料在同一張，一次就能排序 ✅ |
| **修改資料表結構**<br>（例：多加一個欄位） | 每張表都要改一次 | 只要改一份 ✅ |

> 後面會學到「索引」、「唯一值」、「多表串接」這些設定，全部共用一張表，只要設定一次就能套用到所有股票；  
> 一檔一張表的話，每張都要重設一次。

配合加入適當的「索引」，用單一大表查單一支股票的速度跟「一檔一張表」幾乎沒差別，SQLite 處理百萬列也很輕鬆。

> 💡 一個常見的判準：**「同一種事物的不同個體」用「列 (row)」區分，不是「表 (table)」區分。**  
> NVDA 和 AAPL 是同一種事物（股票）的兩個個體，所以它們是 `stock_date` 裡兩組 `stock_id` 不同的列，而不是兩張表。


# SQLite
本次工作坊選擇 SQLite 的原因是：
- **以單一檔案 (`xxx.db`) 儲存**，不需要安裝、設定伺服器，開箱即用
- **Python 內建 `sqlite3` 套件**，免額外安裝
- **SQL 語法和其他關聯式資料庫大致相容**，之後轉換到 MySQL/PostgreSQL 也很容易上手

> SQLite 雖然輕量，卻被廣泛用在手機 App、瀏覽器（Chrome、Firefox）與桌面軟體當中。

使用 SQLite 的一般流程：

``` python
import sqlite3
conn = sqlite3.connect('資料庫名稱.db')   # 資料庫建立/連線
sql = "......"                           # 撰寫 SQL 指令
cursor = conn.execute(sql)               # 執行指令
conn.commit()                            # 會修改到資料的 (INSERT/UPDATE/DELETE) 必須要多寫此行
conn.close()                             # 用完記得關閉連線
```

> `commit()` 就像文件編輯器的「儲存檔案」：如果沒有 commit，改動只存在於當下的連線，不會真正寫入資料庫。

## 建立資料表 (Table)
如同前面介紹的關聯資料庫特性，我們需要事先規劃好**欄位名稱**與**型別**

SQLite 常用型別：

| 型別 | 說明 | 常見用途 |
| --- | --- | --- |
| INTEGER | 整數 | 成交量 |
| REAL | 浮點數 | 股價、均價 |
| TEXT | 文字 | 日期、股票名稱 |
| BLOB | 原始二進位資料 | 圖片、檔案 |
  
``` mysql
CREATE TABLE `資料表` (
    `欄位1`  型別,
    `欄位2`  型別,
    ......
)
```

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db') # 如果資料庫不存在，會自動幫你建立

# 建立 stock_date 資料表
sql_create_table_stock_date = """
CREATE TABLE IF NOT EXISTS `stock_date` (
	`stock_id` TEXT,
	`date`	TEXT,
	`open`	REAL,
	`high`	REAL,
	`low`	REAL,
	`close`	REAL,
	`volume` INTEGER
)
"""
cursor = conn.execute(sql_create_table_stock_date)

# 建立 stock_list 資料表
#     多一個自動編號欄位 `idx` 為 PRIMARY KEY
#     AUTOINCREMENT：自動編號，每新增一筆就 +1，永不重複
sql_create_table_stock_list = """
CREATE TABLE IF NOT EXISTS `stock_list` (
	`idx`	INTEGER PRIMARY KEY AUTOINCREMENT,
	`stock_id`	TEXT,
	`name`	TEXT
)
"""
cursor = conn.execute(sql_create_table_stock_list)

conn.close()


### 為資料表建立唯一索引

預先幫 `stock_date` 這張資料表加上 「**唯一索引 (UNIQUE INDEX)**，避免重複寫入。  
我們的資料是每檔股票的歷史每日資訊，也就是每日每張股票僅為有一筆資料，  
代表`stock_date` 資料表中，`(stock_id, date)` 的組合是唯一值。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql_create_ui = """
    CREATE UNIQUE INDEX IF NOT EXISTS `id_date` ON `stock_date`(`stock_id`, `date`)
"""
conn.execute(sql_create_ui)
conn.commit()
conn.close()


## CRUD

資料表已經建好了，接下來開始用 SQL 對裡面的資料做 CRUD。

### Read 讀取資料（簡易）

最簡單的查詢語法是「把整張表的內容全部撈出來」：

``` mysql
SELECT * FROM `資料表`
```

> 這裡先做簡易版本的操作，這樣等下在做 Create 章節的程式碼時，就能馬上用它來確認資料的改變狀況。
> 完整的 `SELECT` 語法（條件篩選、排序、取前 N 筆...）會在 Create 之後再深入。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = "SELECT * FROM `stock_date`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)

sql = "SELECT * FROM `stock_list`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)

conn.close()

# 程式沒有錯誤：代表 stock_date, stock_list 資料表存在
# 但沒有任何結果是因為 stock_date, stock_list 這幾張資料表還沒有任何一筆資料

### Create 新增資料
``` mysql
INSERT INTO `資料表` (`欄位1`, `欄位2`, ...)
VALUES (值1, 值2, ...)
```
- 文字 (TEXT) 要用單引號 `'` 包起來；數值 (INTEGER / REAL) 則不用
- 欄位順序可以跟建表時不同，只要「欄位名稱」和 `VALUES` 後面的值**一一對應**即可
- 沒有在 `INSERT` 中列出的欄位，會是 `NULL`（除非建表時有指定預設值）

#### 新增資料到 `stock_list`

剛才我們建立了資料表 `stock_list`，有三個欄位
- `idx`：自動編號 (AUTOINCREMENT) — **不用也不該手動指定**，資料庫會自動填入流水號
- `stock_id`：股票代號
- `name`：股票名稱

現在我們要將股票名稱、股票代號的資訊加入資料庫：

| 股票代號 | 股票名稱 |
| --- | --- |
| NVDA | NVIDIA |
| AAPL | APPLE |
| GOOGL | GOOGLE |

In [ ]:
import sqlite3

def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

sql_ins = """
    INSERT INTO `stock_list` (`stock_id`, `name`)
    VALUES ( 'NVDA' ,'NVIDIA' )
"""
insert_safe(sql_ins)

sql_ins = """
    INSERT INTO `stock_list` (`stock_id`, `name`)
    VALUES ( 'AAPL' ,'APPLE' )
"""
insert_safe(sql_ins)

sql_ins = """
    INSERT INTO `stock_list` (`stock_id`, `name`)
    VALUES ( 'GOOGL' ,'GOOGLE' )
"""
insert_safe(sql_ins)

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = "SELECT * FROM `stock_list`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

#### 新增資料到 `stock_date`

- 原始資料沒有股票代號，要記得一併寫入 (`NVDA` 與 `AAPL`)
- 雖然 SQLite 對型別比較寬容，但養成好習慣：INTEGER 欄位用 `int()`、REAL 欄位用 `float()` 先轉型再寫入
- 可搭配 Python 的 `for` 迴圈與字串格式化批次新增，一次只連線一次、一次 commit

In [ ]:
import sqlite3
def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

datas_NVDA = [
    ['2026-04-20', '199.98', '202.17', '197.84', '202.06', '119381400'],
    ['2026-04-21', '202.13', '202.75', '199', '199.88', '107945302'],
    ['2026-04-22', '200.99', '202.5', '199', '202.5', '107501042'],
    ['2026-04-23', '202.46', '203.83', '197.22', '199.64', '113561830'],
    ['2026-04-24', '199.96', '210.965', '199.81', '208.26', '166563954']
]
datas_AAPL = [
    ['2026-04-20', '270.33', '274.27', '270.29', '273.05', '36590200'],
    ['2026-04-21', '271.5', '272.8', '265.4', '266.17', '50209800'],
    ['2026-04-22', '267.82', '273.74', '266.87', '273.17', '43249204'],
    ['2026-04-23', '275.05', '275.77', '271.65', '273.43', '33399639'],
    ['2026-04-24', '272.76', '273.06', '269.67', '270.11', '20541211']
]
def ins_stock_data(stock_id, datas):
    for row in datas:
        d, o, h, l, c, v = row
        sql_ins = """
            INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
            VALUES ( '{}' ,'{}', {}, {}, {}, {}, {} )
        """.format(stock_id, d, float(o), float(h), float(l), float(c), int(v))
        insert_safe(sql_ins)

ins_stock_data('NVDA', datas_NVDA)
ins_stock_data('AAPL', datas_AAPL)

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = "SELECT * FROM `stock_date`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

可用迴圈改寫新增股票資訊的程式碼：

In [ ]:
import sqlite3

def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

stock_infos = [
    ['NVDA' ,'NVIDIA'],
    ['AAPL' ,'APPLE'],
    ['GOOGL', 'GOOGLE']
]

for stock_id, stock_name in stock_infos:
    sql_ins = f"""
        INSERT INTO `stock_list` (`stock_id`, `name`)
        VALUES ( '{stock_id}' ,'{stock_name}' )
    """
    insert_safe(sql_ins)


In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = "SELECT * FROM `stock_list`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

#### [練習] 新增 NVDA 的每日股票資料
請多增加幾筆 NVDA 的股票資訊 到 `stock_list`

In [ ]:
import sqlite3

def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

def ins_stock_data(stock_id, datas):
    for row in datas:
        d, o, h, l, c, v = row
        sql_ins = """
            INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
            VALUES ( '{}' ,'{}', {}, {}, {}, {}, {} )
        """.format(stock_id, d, float(o), float(h), float(l), float(c), int(v))
        insert_safe(sql_ins)

insert_datas_NVDA = [
    ['2026-04-16', '197.43', '199.85', '195.81', '198.35', '134012900'],
    ['2026-04-17', '199.9', '201.7', '199.27', '201.68', '160324417']
]

ins_stock_data(_____, ____)

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = "SELECT * FROM `stock_date`"
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

### Read 讀取資料（進階：篩選與排序）

前面我們用過 `SELECT * FROM 資料表` 把整張表撈出來。  
但實務上常常只想要「特定條件的某幾欄」，這時就需要更完整的語法：

``` mysql
SELECT `欄位1`, `欄位2`, ...
FROM `資料表`
WHERE 條件1 and/or 條件2 ....
ORDER BY `欄位1` 排序方式1, `欄位2` 排序方式2
LIMIT 筆數限制
```

- **顯示欄位**
    - 全選資料表內所有欄位：`SELECT *`
    - 可自訂新欄位（運算符號和 Python 相同）：`SELECT 欄位1, (欄位1 - 欄位2) * 100 / 欄位2`
    - 可自訂欄位順序（不需要和資料表一致）
- **條件**
    - 數值比較符號大致上和 Python 相同
        - 比較兩邊相等：`=`（注意！SQL 用一個等號，不是 `==`）
        - 沒有整除、次方運算子（需用[函式](https://www.w3schools.com/sql/sql_ref_mysql.asp)）
- **排序方式**
    - `DESC` 倒序（大 → 小）
    - `ASC` 正序（小 → 大，可省略不寫）
- **筆數限制**
    - `LIMIT 5` 只取前 5 筆，常配合 `ORDER BY` 做 Top-N 查詢


In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_list`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `stock_id`,`date`,`open`,`close`,`high`,`low`,`volume`, `volume`/1000
    FROM `stock_date`
    WHERE `date` = '2026-04-23' or `volume` >= 110000000
    ORDER BY `stock_id` DESC, `date`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

#### [練習] NVIDIA 股價資料整理與最高價篩選
- 列出 NVIDIA (NVDA) 2026-04-21 ~ 2026-04-24 的每日資訊
    - 日期
    - 開盤、收盤、最高、最低
    - **均價 CDP**：`(最高 + 最低 + 2 * 收盤) / 4`
- 列出擁有最大「最高價」的兩筆資料

> 提示：`ORDER BY` 搭配 `LIMIT` 可以快速取出最大的幾筆。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `date`, `open`, `____`, `____`, `low`, (`___`+`low`+2*`close`)/4
    FROM `stock_date`
    WHERE `stock_id` = '____' and `date` >= '____' and `date` <= '_____'
    ORDER BY `date`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT `stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`
    FROM `stock_date`
    ORDER BY `___` ____
    LIMIT __
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

### Update 更新資料
``` mysql
UPDATE `資料表`
SET `欄位1`=值1, `欄位2`=值2, ...
WHERE 條件1 and/or 條件2 ....
```

- `SET` 可以同時更新多個欄位
- 新值也可以參考舊值：例如美股的購買單位是股，而台股一張是1000股，要把股的單位換算成張，成交量需要除以 1000，`SET volume = volume / 1000`

> ⚠️ 如果忘了寫 `WHERE`，會把**整張表**的該欄位全部改成新值。  
> 養成習慣：執行 `UPDATE` 前，先用相同條件的 `SELECT` 確認範圍是否正確。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql_upd = """
    UPDATE `stock_list`
    SET `name`= 'Alphabet Inc'
    WHERE `stock_id` = 'GOOGL'
"""
cursor = conn.execute(sql_upd)
cursor = conn.commit()
conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_list`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

### Delete 刪除資料
``` mysql
DELETE
FROM `資料表`
WHERE 條件1 and/or 條件2 ....
```

> ⚠️ 同樣地，忘了寫 `WHERE` 等於清空整張表。  
> **刪除前先用 `SELECT` 看看條件對不對**，再換成 `DELETE`。

In [ ]:
# 用內容篩選：WHERE stock_id = 'NVDA'
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_list`
    WHERE `stock_id` = 'NVDA'
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

In [ ]:
# 用主鍵 idx 鎖定：WHERE idx = 1
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_list`
    WHERE `idx` = 1
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

In [ ]:
# 確認過 idx=1 是要刪的那筆後，再做 DELETE
import sqlite3
conn = sqlite3.connect('stocks.db')
sql_del = """
    DELETE
    FROM `stock_list`
    WHERE `idx` = 1
"""
conn.execute(sql_del)
conn.commit()
conn.close()

In [ ]:
# 看看刪除結果
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_list`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

### 處理重複資料
#### 三種常見處理做法

- **新增前先檢查**：先用 `SELECT` 查這筆資料是否已存在，再決定要不要 `INSERT`
- **事後清理**：每筆資料有唯一編號（就是我們的 `idx`），找出重複的 `idx`、用 `DELETE` 留一筆刪其他，也就是我們在 `stock_list` 的處理方式。
- **建立唯一值組合 (UNIQUE INDEX)**：事先定義「哪幾個欄位組合起來必須唯一」，重複插入時資料庫會直接擋下
    - 要先思考哪些欄位組合起來是唯一值
    - `stock_date`：`stock_id + date` 是唯一組合（同一支股票同一天只會有一筆）— 這就是我們前面預先建立的索引
    - `stock_list`：`stock_id` 本身就該唯一（一支股票只該有一筆）

> UNIQUE INDEX 是最省事的方式，但前提是你已經很清楚資料的唯一性規則。

In [ ]:
# 繼續找尋有哪些有重複的資料
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_list`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql_del = """
    DELETE
    FROM `stock_list`
    WHERE `idx` in (2, 3)
"""
conn.execute(sql_del)
conn.commit()
conn.close()

In [ ]:
import sqlite3

def insert_safe(sql_ins):
    conn = sqlite3.connect('stocks.db')
    try:
        conn.execute(sql_ins)
        conn.commit()
        print("新增成功")
    except sqlite3.IntegrityError as e:
        print(f"違反資料庫目前的限制，不新增此筆資料")
    conn.close()

# 嘗試 1：插入一筆已經存在的 NVDA
sql_ins = """
    INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
    VALUES ( 'NVDA' ,'2026-04-23', 202.46, 203.83, 197.22, 199.64, 113561830 )
"""
print("嘗試 1：")
insert_safe(sql_ins)


# 嘗試 2：stock_id, date 不變，更改 open 值
# （因為 UNIQUE 是設在 (stock_id, date) 上，跟 open 無關）
sql_ins2 = """
    INSERT INTO `stock_date` (`stock_id`, `date`, `open`, `high`, `low`, `close`, `volume`)
    VALUES ( 'NVDA' ,'2026-04-23', 1202.46, 203.83, 197.22, 199.64, 113561830 )
"""
print("嘗試 2：")
insert_safe(sql_ins)

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT *
    FROM `stock_date`
"""
cursor = conn.execute(sql)
for row in cursor.fetchall():
    print(row)
conn.close()

上一場活動的金融爬蟲，每天會抓取當日新出來的收盤、成交量等資料。

- **存進資料庫**：每天只要把新抓到的那幾筆 `INSERT` 即可；就算程式不小心跑了兩次，前面為 `stock_date` 預先建立的 `UNIQUE INDEX`（`stock_id + date`）也會幫你擋掉重複
- **寫進試算表**：不管是靠程式寫入還是手動複製貼上，每次都得先把整份檔案讀一輪、逐列比對，才能確認這筆是不是已經存在

> 讓資料庫幫你把「唯一性」顧好，正是關聯式資料庫在日常自動化流程裡最省事又可靠的地方。


## INNER JOIN 多表串接
為什麼需要多張資料表？

- **一對多關係**：一個客戶會有很多訂單、一部電影會有很多演員，這些資料沒辦法塞進一格裡
- **資料只在一個地方管理**：把重複出現的資訊抽成獨立表格，不用每筆都抄一次
    - `stock_list`：股票代碼、股票名稱（每支股票一筆）
    - `stock_date`：股票代碼、日期、各交易原始資訊（每支股票每天一筆）
    - 若「Google」改名「Alphabet Inc」，只需改 `stock_list` 一列，不用改幾千筆 `stock_date`

**JOIN** 就是在查詢時，把這些分散的資料表「黏」回一起，透過**共通欄位**（通常是代號/ID）做對應。

``` mysql
SELECT `資料表n`.`欄位1`, `資料表n`.`欄位2`, ...
FROM `資料表1`
INNER JOIN `資料表2` ON 關聯條件1 and/or 條件2
WHERE 條件1 and/or 條件2 ....
ORDER BY 排序方式1, 排序方式2
LIMIT 筆數限制
```

- **欄位前記得加資料表名稱**（`資料表.欄位`），避免不同表有同名欄位（例如兩張表都有 `id`）造成歧義
- **`ON` 是關聯條件**，通常是「A 表的某欄位 = B 表的某欄位」
- 可以連續 `INNER JOIN` 多張表。
- 只有「兩邊都對得上」的資料才會出現在結果中，這正是 **INNER** 的意思

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT
        `stock_list`.`name`,
        `stock_date`.*
    FROM `stock_date`
    INNER JOIN `stock_list` ON `stock_date`.`stock_id` = `stock_list`.`stock_id`
    ORDER BY `stock_date`.`stock_id`, `stock_date`.`date`
"""
cursor = conn.execute(sql)
names = tuple(description[0] for description in cursor.description)
print(names)
for row in cursor.fetchall():
    print(row)
conn.close()

### 多對多關係
一支股票可能同時屬於多個分類（NVIDIA 既是「科技股」也是「半導體股」），一個分類也會包含很多股票。這時候會設計三張表：

- `stock_list`：股票代碼、股票名稱
- `category`：分類代號、分類名稱（科技股、金融股、消費股…）
- `stock_category`：哪支股票屬於哪個分類（多對多的**橋樑表**）

要列出「所有科技股的股票代碼與名稱」，就需要把這三張表透過 JOIN 串起來。

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')

# 分類：代號對應分類名稱
conn.execute("""
CREATE TABLE `category` (
    `category_id`    INTEGER,
    `category_name`  TEXT
)
""")

# 橋樑表：哪支股票屬於哪個分類（多對多）
conn.execute("""
CREATE TABLE `stock_category` (
    `stock_id`     TEXT,
    `category_id`  INTEGER
)
""")

conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')

# 增加 6 支常見美股
stock_list = [
    ('MSFT',  'Microsoft'),
    ('JPM',   'JPMorgan Chase'),
    ('WMT',   'Walmart'),
    ('KO',    'Coca-Cola'),
    ('SBUX',  'Starbucks'),
]
conn.executemany("INSERT INTO `stock_list` (`stock_id`, `name`) VALUES (?, ?)", stock_list)

# 5 個分類
category = [
    (1, '科技股'),
    (2, '半導體'),
    (3, '金融股'),
    (4, '消費股'),
    (5, '標普500'),
]
conn.executemany("INSERT INTO `category` VALUES (?, ?)", category)

# 多對多：一支股票可以對應多個分類
stock_category = [
    ('NVDA',  1), ('NVDA',  2), ('NVDA',  5),   # NVIDIA：科技、半導體、標普500
    ('AAPL',  1), ('AAPL',  5),                 # Apple：科技、標普500
    ('GOOGL', 1), ('GOOGL', 5),                 # Alphabet：科技、標普500
    ('MSFT',  1), ('MSFT',  5),                 # Microsoft：科技、標普500
    ('JPM',   3), ('JPM',   5),                 # JPMorgan：金融、標普500
    ('WMT',   4), ('WMT',   5),                 # Walmart：消費、標普500
    ('KO',    4), ('KO',    5),                 # Coca-Cola：消費、標普500
    ('SBUX',  4), ('SBUX',  5),                 # Starbucks：消費、標普500
]
conn.executemany("INSERT INTO `stock_category` VALUES (?, ?)", stock_category)

conn.commit()
conn.close()

In [ ]:
import sqlite3
conn = sqlite3.connect('stocks.db')
sql = """
    SELECT
        `stock_list`.`stock_id`,
        `stock_list`.`name`,
        `category`.`category_name`
    FROM `stock_list`
    INNER JOIN `stock_category` ON `stock_list`.`stock_id` = `stock_category`.`stock_id`
    INNER JOIN `category`       ON `stock_category`.`category_id` = `category`.`category_id`
    WHERE `category`.`category_name` = '科技股'
    ORDER BY `stock_list`.`stock_id`
"""
cursor = conn.execute(sql)
names = tuple(description[0] for description in cursor.description)
print(names)
for row in cursor.fetchall():
    print(row)
conn.close()

## 學習資源
- [w3school SQL 教學](https://www.w3schools.com/sql/default.asp)：每個語法都有可互動的範例
- [深入淺出 SQL](https://www.tenlong.com.tw/products/9789866840166)：從需求到設計、非常適合入門的書

### 延伸主題
- **LEFT JOIN / RIGHT JOIN**：即使某一邊沒有對應資料也要保留
- **GROUP BY / HAVING**：依群組做統計（每支股票最高收盤價、每年電影數量⋯）
- **Transaction (交易)**：把多個操作包成一組，全部成功才生效，否則全部回滾

## 附錄：將 csv 資料轉換成串列
金融數據工作坊系列活動中，
將 [爬蟲抓取的 csv](https://python-practical-workshop.github.io/crawler/) 用以下程式轉換成串列，  
就可利用我們前面學到的語法與程式將資料新增到資料庫


In [ ]:
import csv

def load_csv(filename):
    with open(filename, "r") as fi:
        reader = csv.reader(fi)
        data = list(reader)
    return data

data = load_csv("NVDA_fmp.csv")
print(data[0:5])

data_without_header = data[1:]
print(data_without_header[0:5])

# Flask

## 為什麼要學架網站？
- **跨平台、跨裝置**：只要能上網，不管手機、平板、電腦都能使用
- **不需要安裝**：使用者打開瀏覽器就能用，不用先裝 Python 或 App
- **爬蟲的反向練習**：爬蟲是在抓別人的網站；自己架過一次網站後，更能理解網頁如何產生、資料怎麼傳遞，爬起來也會更順手

## 為什麼用 Flask？
- **容易上手**：少少幾行就能跑起一個網站
- **輕量化**：沒有多餘的設定，想加什麼功能再自己裝
- **生態完整**：Jinja2 模板、Flask-Bootstrap、Flask-SQLAlchemy 等擴充套件應有盡有

> 類似定位的還有 **FastAPI**（新一代、以 API 為主）、**Django**（大而全的企業級框架），未來可依需求選用。

> 💡 網頁是把資料**分享給人 / 其他系統**的一種形式。
> 第三場「資料分析與視覺化」會學另一種分享形式 — 把資料變成圖表，分享「洞察」給看圖的人。
> 兩者共用同一個資料庫，只是呈現方式不同。


## 申請 Ngrok Authtoken
https://dashboard.ngrok.com/get-started/your-authtoken


In [ ]:
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN"

In [ ]:
!pip install pyngrok flask --quiet

## 基本 route
**Route（路由）** 就是「**網址** → **要執行的函式**」的對應關係。

使用者在瀏覽器輸入網址 → Flask 看到這個網址 → 找到對應的函式 → 執行後把結果（通常是 HTML 字串）回傳給瀏覽器顯示。

下面的範例中：
- `/` 對應 `home()`
- `/stock` 對應 `stock()`
- 函式回傳什麼內容，瀏覽器就顯示什麼

> `@app.route(...)` 這種寫法叫做 **decorator（裝飾器）**，你可以先把它當成「為函式貼上一張網址標籤」的意思。

In [ ]:
from flask import Flask
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

@app.route("/")
def home():
    resp = "Welcome to Stock Board <br> <a href='/stock_list'>進入股票列表頁</a>"
    return resp

@app.route("/stock_list")
def stock_list():
    resp = """
        <h1>股票列表</h1>
        <a href='/'>回首頁</a>
        <table>
            <thead>
                <tr>
                    <th>股票代碼</th>
                    <th>股票名</th>
                    <th>股票頁</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td>NVDA</td>
                    <td>NVIDIA</td>
                    <td><a href="/stock">進入股票頁</a></td>
                </tr>
                <tr>
                    <td>AAPL</td>
                    <td>APPLE</td>
                    <td><a href="/stock">進入股票頁</a></td>
                </tr>
            </tbody>
        </table>
    """
    return resp

@app.route("/stock")
def stock():
    stock_id = "NVDA"

    resp = """
        <h1>股票代碼: {}</h1>
        <a href='/stock_list'>回股票列表頁</a>
        <table>
            <thead>
                <tr>
                    <th>日期</th>
                    <th>開盤價</th>
                    <th>最高價</th>
                    <th>最低價</th>
                    <th>收盤價</th>
                    <th>成交量</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td>2026-04-22</td>
                    <td>200.99</td>
                    <td>202.5</td>
                    <td>199</td>
                    <td>202.5</td>
                    <td>107501042</td>
                </tr>
                <tr>
                    <td>2026-04-23</td>
                    <td>202.46</td>
                    <td>203.83</td>
                    <td>197.22</td>
                    <td>199.64</td>
                    <td>113561830</td>
                </tr>
                <tr>
                    <td>2026-04-24</td>
                    <td>199.96</td>
                    <td>210.965</td>
                    <td>199.81</td>
                    <td>208.26</td>
                    <td>166563954</td>
                </tr>
            </tbody>
        </table>
    """.format(stock_id)
    return resp

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

實際上我們不會把會變動的資料，直接寫死在 HTML 之中

In [ ]:
from flask import Flask
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

@app.route("/")
def home():
    resp = "Welcome to Stock Board <br> <a href='/stock_list'>進入股票列表頁</a>"
    return resp

@app.route("/stock_list")
def stock_list():
    datas = [
        ['NVDA',  'NVIDIA'],
        ['AAPL',  'APPLE'],
    ]
    html_data = ""
    for row in datas:
        stock_id, name = row
        html_row = """
            <tr>
                <td>{}</td>
                <td>{}</td>
                <td><a href="/stock">進入股票頁</a></td>
            </tr>""".format(stock_id, name)
        html_data += html_row
    resp = """
        <h1>股票列表</h1>
        <a href='/'>回首頁</a>
        <table>
            <thead>
                <tr>
                    <th>股票代碼</th>
                    <th>股票名</th>
                    <th>股票頁</th>
                </tr>
            </thead>
            <tbody>
                {}
            </tbody>
        </table>
    """.format(html_data)
    return resp


@app.route("/stock")
def stock():
    stock_id = "NVDA"
    datas_NVDA = [
        ['2026-04-22', 200.99, 202.5, 199, 202.5, 107501042],
        ['2026-04-23', 202.46, 203.83, 197.22, 199.64, 113561830],
        ['2026-04-24', 199.96, 210.965, 199.81, 208.26, 166563954]
    ]
    html_data = ""
    for row in datas_NVDA:
        d, o, h, l, c, v = row
        html_row = """
            <tr>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
                <td>{}</td>
            </tr>""".format(d, o, h, l, c, v)
        html_data+=html_row
    resp = """
        <h1>股票代碼: {}</h1>
        <a href='/stock_list'>回股票列表頁</a>
        <table>
            <thead>
                <tr>
                    <th>日期</th>
                    <th>開盤價</th>
                    <th>最高價</th>
                    <th>最低價</th>
                    <th>收盤價</th>
                    <th>成交量</th>
                </tr>
            </thead>
            <tbody>
                {}
            </tbody>
        </table>
    """.format(stock_id,html_data)
    return resp

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

## Template 模板
前面的寫法有個明顯的問題：**HTML 跟 Python 程式碼混在一起**，  
當頁面變複雜、或設計師想換版型時，就會非常難維護。

**Template** 的核心概念是**把版型和資料分離**：
- HTML 單獨寫在一個檔案裡，裡面預留「填入資料」的位置
- Python 只負責準備資料、呼叫 `render_template` 把兩者合併

做法：
- 新建資料夾 `templates`（名稱不能改，Flask 預設從這裡找）
- 在 `templates` 資料夾中，新增檔案 `stock.html`, `stock_list.html`

Flask 使用的模板語法叫 **Jinja2**：
- `{{ 變數 }}`：把變數的值填進來
- `{% for ... %} ... {% endfor %}`：迴圈
- `{% if ... %} ... {% endif %}`：條件判斷

In [ ]:
home_html = "Welcome to Stock Board <br> <a href='/stock_list'>進入股票列表頁</a>"

In [ ]:
stock_list_html = """
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>股票代碼</th>
            <th>股票名</th>
            <th>股票頁</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td><a href="/stock">進入股票頁</a></td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

In [ ]:
stock_html = """
<h1>股票代碼: {{ stock_id }}</h1>
<a href='/stock_list'>回股票列表頁</a>
<table>
    <thead>
        <tr>
            <th>日期</th>
            <th>開盤價</th>
            <th>最高價</th>
            <th>最低價</th>
            <th>收盤價</th>
            <th>成交量</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td>{{ row[2] }}</td>
            <td>{{ row[3] }}</td>
            <td>{{ row[4] }}</td>
            <td>{{ row[5] }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

with open("templates/home.html","w") as fo:
    fo.write(home_html)

with open("templates/stock_list.html","w") as fo:
    fo.write(stock_list_html)

with open("templates/stock.html","w") as fo:
    fo.write(stock_html)

對照下面 Python 端的寫法：`render_template("stock.html", stock_id=stock_id, datas=datas)`，  
前面的 `stock_id`、`datas` 是「模板內的變數名稱」，後面則是「對應要傳入的 Python 值」。

In [ ]:
from flask import Flask, render_template
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

@app.route("/")
def home():
    return render_template("home.html")

@app.route("/stock_list")
def stock_list():
    datas = [
        ['NVDA',  'NVIDIA'],
        ['AAPL',  'APPLE'],
    ]
    return render_template("stock_list.html", datas = datas)

@app.route("/stock")
def stock():
    stock_id = "NVDA"
    datas = [
        ['2026-04-22', 200.99, 202.5, 199, 202.5, 107501042],
        ['2026-04-23', 202.46, 203.83, 197.22, 199.64, 113561830],
        ['2026-04-24', 199.96, 210.965, 199.81, 208.26, 166563954]
    ]
    return render_template("stock.html", stock_id=stock_id, datas = datas)

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

## Parameter 網址參數
目前 `/stock` 永遠只顯示 NVDA 一支股票，代號是寫死的。  
要讓使用者能看任意一支股票，就需要**把股票代號從網址傳進來**。

Flask 支援兩種傳參方式：

| 方式 | 網址範例 | Flask 語法 | 適用情境 |
| --- | --- | --- | --- |
| **路徑參數 (Path)** | `/stock/NVDA` | `@app.route("/stock/<stock_id>")` + 函式接收 `stock_id` | 必填、屬於「對誰做事」的資訊 |
| **查詢字串 (Query)** | `/stock?stock_id=NVDA` | `request.args.get('stock_id')` | 選填、可多可少、可排列組合的過濾條件 |

> 兩者功能上都能做到，實務上常見的搭配是：「**必填的 id 放路徑，可選的過濾條件放 query string**」，  
> 例如 `/stock/NVDA?date_start=2026-04-20&date_end=2026-04-24`。

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

stock_list_html = """
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>股票代碼</th>
            <th>股票名</th>
            <th>股票頁(路徑)</th>
            <th>股票頁(參數)</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td><a href="/stock/{{ row[0] }}">進入股票頁(路徑)</a></td>
            <td><a href="/stock?stock_id={{ row[0] }}">進入股票頁(參數)</a></td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

with open("templates/stock_list.html","w") as fo:
    fo.write(stock_list_html)

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id):
    datas = [
        ['2026-04-22', 200.99, 202.5, 199, 202.5, 107501042],
        ['2026-04-23', 202.46, 203.83, 197.22, 199.64, 113561830],
        ['2026-04-24', 199.96, 210.965, 199.81, 208.26, 166563954]
    ]
    return datas

@app.route("/")
def home():
    return render_template("home.html")

@app.route("/stock_list")
def stock_list():
    datas = [
        ['NVDA',  'NVIDIA'],
        ['AAPL',  'APPLE'],
    ]
    return render_template("stock_list.html", datas = datas)

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

## 串接 SQLite 資料庫
到目前為止，`get_stock_datas()` 回傳的都是**寫死的假資料**。  
現在我們把它改寫成從前面建立的 `stocks.db` 真實查詢。

關鍵就是在 Flask 路由的函式裡面：連線資料庫 → 執行 SELECT → 回傳結果給模板。  
這也讓**每支股票只要一次程式碼，就能查任意代號**。

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id):
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}'
        ORDER BY `date`
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

def get_stock_list():
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `stock_id`, `name`
        FROM `stock_list`
        WHERE `stock_id` in ('NVDA', 'AAPL')
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

@app.route("/")
def home():
    return render_template("home.html")

@app.route("/stock_list")
def stock_list():
    return render_template("stock_list.html", datas = get_stock_list())

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

### [練習] 增加 `date_start` 參數
- 增加一個參數 `date_start`
    - 如果走 `get_stock_path`，url 設計成 `/stock/NVDA/YYYY-MM-DD`
    - 如果走 `get_stock_param`，url 設計成 `/stock?stock_id=NVDA&date_start=YYYY-MM-DD`
- 先把收到的 `date_start` `print` 出來確認有正確接到
- 把 `date_start` 當作 SQL 查詢條件：搜尋此日期（包含）之後的每日資訊

#### 提示：  
- path 版的網頁路徑和函式傳入的參數要增加 `date_start`   
- query 版的 `date_start` 則是要用 request.args.get 取得，是在函式內增加一行程式碼。
- 在 SQLite 是以 TEXT 儲存，WHERE 條件記得用單引號：`` `date` >= 'YYYY-MM-DD' ``
  
  
> 想想看：如果 `date_start` 沒傳入（為 `None`）時，SQL 該怎麼組才不會出錯？

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

home_with_date_html = "Welcome to Stock Board <br> <a href='/stock_list?date_start=2026-04-22'>進入股票列表頁</a>"

stock_list_with_date_html = """
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>股票代碼</th>
            <th>股票名</th>
            <th>股票頁(路徑)</th>
            <th>股票頁(參數)</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td><a href="/stock/{{ row[0] }}/{{ date_start }}">進入股票頁(路徑)</a></td>
            <td><a href="/stock?stock_id={{ row[0] }}&date_start={{ date_start }}">進入股票頁(參數)</a></td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

with open("templates/home_with_date.html","w") as fo:
    fo.write(home_with_date_html)

with open("templates/stock_list_with_date.html","w") as fo:
    fo.write(stock_list_with_date_html)

In [ ]:
from flask import Flask, render_template, request
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)

def get_stock_datas(stock_id, date_start):
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}' and `date` >= '{}'
        ORDER BY `date`
    """.format(stock_id, ______)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    conn.close()
    return datas

def get_stock_list():
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `stock_id`, `name`
        FROM `stock_list`
        WHERE `stock_id` in ('NVDA', 'AAPL')
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

@app.route("/")
def home():
    return render_template("home_with_date.html")

@app.route("/stock_list")
def stock_list():
    date_start = request.args.get('_______')
    return render_template("stock_list_with_date.html", date_start=_______, datas = get_stock_list())

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock/<stock_id>/<_______>")
def get_stock_date_path(stock_id, _______):
    return render_template("stock.html", stock_id=stock_id, date_start=_______, datas=get_stock_datas(stock_id, _______))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    date_start = request.args.get('_______')
    return render_template("stock.html", stock_id=stock_id, date_start=_______, datas=get_stock_datas(stock_id, _______))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

### SQL Injection
SQL Injection 是攻擊者利用應用程式對使用者輸入處理不當，  
將惡意 SQL 片段「拼接」到原本的查詢語句中，使資料庫執行非預期的操作。

e.g.
`date_start` 輸入
```
2026-04-22'; DELETE FROM `stock_date`;--
```

正式環境請用 `?` 參數化查詢：  

In [ ]:
sql = """
    SELECT `date`, `open`, `high`, `low`, `close`, `volume`
    FROM `stock_date`
    WHERE `stock_id` = ? AND `date` = ?
    ORDER BY `date`
"""
cursor = conn.execute(sql, (stock_id, date_start))

## 讓網頁變好看：Bootstrap / DataTables
目前做出來的網站雖然會動，但是**長得很陽春**，沒有設計感。  
在不學完整前端的前提下，最快的做法是套現成的前端框架與元件：

### [Bootstrap](https://getbootstrap.com/)
- 包含 HTML、CSS、JavaScript 的前端框架，內建許多已經設計好的元件（按鈕、表格、導覽列…）
- 不用自己從零寫 CSS，就能讓頁面看起來有質感
- **響應式設計 (RWD)**：同一份程式碼，桌機、平板、手機都能自動適應

### [DataTables](https://datatables.net/)
- jQuery 套件，把普通表格升級成「**可排序、可搜尋、可分頁**」的互動表格
    - jQuery：JavaScript 函式庫，簡化 HTML 與 JavaScript 之間的操作
- 會在前端自動重新排序資料，所以**原始 SQL 的 `ORDER BY` 只會影響初始載入的順序**

In [ ]:
!pip install flask-bootstrap

### [Templates 下載](https://github.com/MarsW/slides/tree/master/codes/flask/stock/templates)
- `b_stock.html`：套用 Bootstrap 樣式的**股票頁**（純表格）
- `bd_stock.html`：套用 Bootstrap + DataTables 的**股票頁**（互動式表格）

> 可以打開這三個檔案比較一下：除了 `<head>` 引入的 CSS/JS 之外，和原本的 `stock.html` 只差在加了幾個 class。

In [ ]:
import os
import requests
file_urls = [
    "https://raw.githubusercontent.com/Python-Practical-Workshop/sql_flask/refs/heads/master/templates/b_stock.html",
    "https://raw.githubusercontent.com/Python-Practical-Workshop/sql_flask/refs/heads/master/templates/bd_stock.html"
]

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

for file_url in file_urls:
    file_name = file_url.split("/")[-1]
    response = requests.get(file_url)
    with open(f"templates/{file_name}","w") as fo:
        fo.write(response.text)


In [ ]:
from flask import Flask, render_template, request
from flask_bootstrap import Bootstrap
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)
bootstrap = Bootstrap(app)

def get_stock_datas(stock_id):
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}'
        ORDER BY `date`
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

def get_stock_list():
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `stock_id`, `name`
        FROM `stock_list`
        WHERE `stock_id` in ('NVDA', 'AAPL')
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

@app.route("/")
def home():
    return render_template("home.html")

@app.route("/stock_list")
def stock_list():
    return render_template("stock_list.html", datas = get_stock_list())

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("b_stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    return render_template("bd_stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

## 把後端當 API 用
到目前為止，我們的 Flask 回傳的都是**HTML 頁面**（字串包 `<html>...`）。  
但 Flask 也可以只回「**純資料**」—— 也就是 **JSON**，不組畫面、不管版型。

這種寫法就是所謂的 **API (Application Programming Interface)**：

| 服務對象 | 回傳格式 | 典型網址 |
| --- | --- | --- |
| 給「人」看 | HTML | `/stock/NVDA` |
| 給「程式」處理 | JSON | `/api/stock/NVDA` |

好處是**同一份資料可以給多種前端使用**：
- 網頁用 JavaScript 使用它
- 行動 App 用它
- 用 Python `requests` 爬取它做分析

彼此只要先講好 JSON 的欄位格式，之後想改版面只動前端、想換資料庫只動後端，**兩邊獨立作業**。

Flask 用 `jsonify` 就能把 Python 的 list / dict 轉成標準 JSON 回傳。

In [ ]:
import os

dir_name = "templates"
if dir_name not in os.listdir():
    os.makedirs(dir_name)

stock_list_with_api_html = """
<a href='/'>回首頁</a>
<table>
    <thead>
        <tr>
            <th>股票代碼</th>
            <th>股票名</th>
            <th>股票頁(路徑)</th>
            <th>股票頁(參數)</th>
            <th>股票API</th>
        </tr>
    </thead>
    <tbody>
        {% for row in datas %}
        <tr>
            <td>{{ row[0] }}</td>
            <td>{{ row[1] }}</td>
            <td><a href="/stock/{{ row[0] }}">進入股票頁(路徑)</a></td>
            <td><a href="/stock?stock_id={{ row[0] }}">進入股票頁(參數)</a></td>
            <td><a href="/api/stock/{{ row[0] }}">進入股票API</a></td>
        </tr>
        {% endfor %}
    </tbody>
</table>
"""

with open("templates/stock_list_with_api.html","w") as fo:
    fo.write(stock_list_with_api_html)

In [ ]:
from flask import Flask, render_template, request, jsonify
from flask_bootstrap import Bootstrap
from pyngrok import ngrok
import sqlite3
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

app = Flask(__name__)
bootstrap = Bootstrap(app)

def get_stock_datas(stock_id):
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `date`, `open`, `high`, `low`, `close`, `volume`
        FROM `stock_date`
        WHERE `stock_id` = '{}'
        ORDER BY `date`
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

def get_stock_list():
    conn = sqlite3.connect('stocks.db')
    sql = """
        SELECT `stock_id`, `name`
        FROM `stock_list`
        WHERE `stock_id` in ('NVDA', 'AAPL')
    """.format(stock_id)
    cursor = conn.execute(sql)
    datas = cursor.fetchall()
    return datas

@app.route("/")
def home():
    return render_template("home.html")

@app.route("/stock_list")
def stock_list():
    return render_template("stock_list_with_api.html", datas = get_stock_list())

@app.route("/stock/<stock_id>")
def get_stock_path(stock_id):
    return render_template("b_stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

@app.route("/stock")
def get_stock_param():
    stock_id = request.args.get('stock_id')
    return render_template("bd_stock.html", stock_id=stock_id, datas = get_stock_datas(stock_id))

# 給「程式」用：回傳 JSON
@app.route("/api/stock/<stock_id>")
def api_stock(stock_id):
    datas = get_stock_datas(stock_id)
    result = []
    for row in datas:
        d, o, h, l, c, v = row
        result.append({
            "date":   d,
            "open":   o,
            "high":   h,
            "low":    l,
            "close":  c,
            "volume": v,
        })
    return jsonify(result)

public_url = ngrok.connect(5000)
print(" * 公開網址:", public_url)

app.run()

還記得金融數據工作坊系列活動中，爬蟲處理回傳是 JSON 的網站嗎？
**它們背後的實作方式，就是你剛剛寫的那幾行 `jsonify()`。**

當時我們是「消費者」，用 `requests.get(url)` 把資料抓下來；  
現在你是「生產者」，也能提供同樣格式的 API 給別人用。

打開 `https://你的ngrok網址/api/stock/NVDA` 看看，畫面就是一份 JSON。  
也可以試著用 `requests` 反過來抓自己：

``` python
import requests
res = requests.get("https://你的ngrok網址/api/stock/NVDA")
print(res.json())
```

> 這個就是後端工程師最核心的能力之一：**把資料變成別人能用的服務**。

## 學習資源
- [Flask Web 開發：基於 Python 的 Web 應用開發實戰](https://www.tenlong.com.tw/products/9787115373991)
- 學習地圖
    - [前端](https://softnshare.com/webfrontendprogrammer/)：進一步學習 HTML / CSS / JavaScript
    - [後端](https://softnshare.com/backenddeveloper/)：資料庫、伺服器、API 設計
        - VPS 主機服務：Linode, DigitalOcean, GCP, AWS, Azure
            - [鳥哥的 Linux 私房菜](http://linux.vbird.org/)：Linux 伺服器操作必備

### 延伸主題
- **Flask-SQLAlchemy**：用 Python 物件的方式操作資料庫，不用直接寫 SQL 字串，也能避免 SQL Injection。
- **Flask-Login**：處理使用者登入、權限控管
- **Deploy（部署）**：搭配 Gunicorn + Nginx，把網站部署到正式環境，不再依賴 ngrok